## UAV Intrusion Detection Stacked Ensemble
This notebook uses One-Class SVM, Autoencoder, and Local Outlier Factor as base models, and combines their outputs with a meta-classifier ensemble.

In [7]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import OneClassSVM
from sklearn.neighbors import LocalOutlierFactor
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import classification_report
import os

In [8]:
# Constants
RANDOM_STATE = 1082
base_dir = "../../../Datasets/excel/2dSpoofingDataset/"
real_data_path = "../../../Datasets/excel/2dSpoofingDataset/GPS_Data_Simplified_2D_Feature_Map.xlsx"
file_names = [
    "spoofing-CTGAN-synthetic.csv",
    "spoofing-GaussianCopula-synthetic.csv",
    "spoofing-TVAE-synthetic.csv",
]

In [9]:
# Load real data
try:
    real_df = pd.read_excel(real_data_path)
    print("Loaded real data")
except Exception as e:
    print(f"Could not load real data: {e}")
    real_df = None

if real_df is None or 'Output' not in real_df.columns:
    raise RuntimeError("Real data not loaded or missing 'Output' column.")

X_real = real_df.drop('Output', axis=1)
y_real = real_df['Output']

# Loop through each synthetic dataset
for file_name in file_names:
    file_path = os.path.join(base_dir, file_name)
    try:
        df = pd.read_csv(file_path)
        print(f"\nProcessing {file_name}")
    except FileNotFoundError:
        print(f"Error: The file '{file_path}' was not found.")
        continue
    except Exception as e:
        print(f"An error occurred while loading {file_name}: {e}")
        continue

    if 'Output' not in df.columns:
        print(f"Warning: 'Output' column not found in {file_name}. Skipping.")
        continue

    # Use only benign samples from synthetic for training
    X_train_raw = df[df['Output'] == 0].drop('Output', axis=1)
    # Synthetic labels and all data for stacking/meta-classifier training
    X_synth_full = df.drop('Output', axis=1)
    y_synth_full = df['Output']

    # Normalize using only synthetic benign training data
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_raw)
    X_synth_full_scaled = scaler.transform(X_synth_full)
    X_real_scaled = scaler.transform(X_real)

    # PCA for OC-SVM and LOF (85% variance)
    pca_svm_lof = PCA(n_components=0.85, random_state=RANDOM_STATE)
    X_train_pca_svm_lof = pca_svm_lof.fit_transform(X_train_scaled)
    X_synth_full_pca_svm_lof = pca_svm_lof.transform(X_synth_full_scaled)
    X_real_pca_svm_lof = pca_svm_lof.transform(X_real_scaled)

    # PCA for Autoencoder (95% variance)
    pca_ae = PCA(n_components=0.95, random_state=RANDOM_STATE)
    X_train_pca_ae = pca_ae.fit_transform(X_train_scaled)
    X_synth_full_pca_ae = pca_ae.transform(X_synth_full_scaled)
    X_real_pca_ae = pca_ae.transform(X_real_scaled)

    # One-Class SVM
    ocsvm = OneClassSVM(nu=0.011, gamma=0.000211)
    ocsvm.fit(X_train_pca_svm_lof)
    ocsvm_score_synth = ocsvm.decision_function(X_synth_full_pca_svm_lof)
    ocsvm_score_real = ocsvm.decision_function(X_real_pca_svm_lof)

    # Local Outlier Factor
    lof = LocalOutlierFactor(n_neighbors=1000, novelty=True)
    lof.fit(X_train_pca_svm_lof)
    lof_score_synth = lof.decision_function(X_synth_full_pca_svm_lof)
    lof_score_real = lof.decision_function(X_real_pca_svm_lof)

    # Autoencoder using MLP
    ae = MLPRegressor(hidden_layer_sizes=(3,), activation='relu', max_iter=100, random_state=RANDOM_STATE)
    ae.fit(X_train_pca_ae, X_train_pca_ae)
    recon_synth = ae.predict(X_synth_full_pca_ae)
    recon_real = ae.predict(X_real_pca_ae)
    ae_score_synth = np.mean((X_synth_full_pca_ae - recon_synth) ** 2, axis=1)
    ae_score_real = np.mean((X_real_pca_ae - recon_real) ** 2, axis=1)

    # Stack anomaly scores as features
    stacked_features_synth = np.vstack([ocsvm_score_synth, lof_score_synth, ae_score_synth]).T
    stacked_features_real = np.vstack([ocsvm_score_real, lof_score_real, ae_score_real]).T

    # Meta-classifier (Random Forest) trained on synthetic, evaluated on real
    meta_clf = RandomForestClassifier(random_state=RANDOM_STATE)
    meta_clf.fit(stacked_features_synth, y_synth_full)
    stacked_pred = meta_clf.predict(stacked_features_real)

    print(f"Training on synthetic: {file_name}, testing on real data")
    print(classification_report(
        y_real,
        stacked_pred,
        labels=[0, 1, 2, 3],
        target_names=["benign", "attack1", "attack2", "attack3"],
        zero_division=0
    ))

Loaded real data

Processing spoofing-CTGAN-synthetic.csv


C:\Users\antho\miniforge3\envs\sklearn-env\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (100) reached and the optimization hasn't converged yet.
  warnings.warn(


Training on synthetic: spoofing-CTGAN-synthetic.csv, testing on real data
              precision    recall  f1-score   support

      benign       0.85      0.38      0.53    397825
     attack1       0.19      0.53      0.27     36458
     attack2       0.16      0.38      0.22     44232
     attack3       0.13      0.50      0.21     32015

    accuracy                           0.40    510530
   macro avg       0.33      0.45      0.31    510530
weighted avg       0.70      0.40      0.46    510530


Processing spoofing-GaussianCopula-synthetic.csv


C:\Users\antho\miniforge3\envs\sklearn-env\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (100) reached and the optimization hasn't converged yet.
  warnings.warn(


Training on synthetic: spoofing-GaussianCopula-synthetic.csv, testing on real data
              precision    recall  f1-score   support

      benign       0.78      0.99      0.87    397825
     attack1       0.03      0.00      0.00     36458
     attack2       0.08      0.00      0.00     44232
     attack3       0.04      0.00      0.00     32015

    accuracy                           0.77    510530
   macro avg       0.23      0.25      0.22    510530
weighted avg       0.62      0.77      0.68    510530


Processing spoofing-TVAE-synthetic.csv


C:\Users\antho\miniforge3\envs\sklearn-env\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (100) reached and the optimization hasn't converged yet.
  warnings.warn(


Training on synthetic: spoofing-TVAE-synthetic.csv, testing on real data
              precision    recall  f1-score   support

      benign       0.78      0.97      0.86    397825
     attack1       0.10      0.02      0.04     36458
     attack2       0.16      0.02      0.04     44232
     attack3       0.10      0.01      0.03     32015

    accuracy                           0.76    510530
   macro avg       0.29      0.26      0.24    510530
weighted avg       0.64      0.76      0.68    510530

